# BCSS-WSSS - Stage 1 ResNet38d Multi-Label Classification & CAM Training
## Production-Grade Colab Pro Notebook: 23,422 BCSS-WSSS Training Patches
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khoinguyen59/medical-wsss/blob/master/notebooks/01_Train_Stage1_Classifier_T4.ipynb)


* **Task:** Train Stage 1 ResNet-38d dilated backbone (105M parameters) on 23,422 weakly-supervised breast cancer histopathology patches (BCSS-WSSS).
* **Dedicated Hardware:** NVIDIA Tesla T4 GPU (16GB VRAM, Turing Tensor Cores, Mixed Precision FP16)
* **Precision:** Mixed Precision FP16 (Native Turing Tensor Cores)
* **Loss Formulation:** Multi-Label Soft Margin Loss with Progressive Dropout Attention (PDA) activation for extensive Class Activation Map (CAM) coverage.
* **Checkpoints:** Direct high-speed NVMe SSD storage on Colab VM disk (`/content/checkpoints/`) with instant one-click download (Zero Google Drive dependency).


---
## Step 1: Configure High-Speed Colab Local Storage (/content/checkpoints/) - Zero Drive Required
Configures high-speed local NVMe SSD storage at `/content/checkpoints/` directly on the Colab VM disk.


In [ ]:
import os
import sys
from pathlib import Path

# Check if running inside Google Colab
IN_COLAB = "google.colab" in sys.modules

# Pure Colab Local NVMe Storage (Pure GitHub + Colab, Zero Drive Popup)
if IN_COLAB:
    CHECKPOINT_DIR = Path("/content/checkpoints")
    print("[STORAGE] Using high-speed Colab VM local disk at /content/checkpoints")
else:
    CHECKPOINT_DIR = Path("./checkpoints")

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
STAGE1_LATEST_CKPT = CHECKPOINT_DIR / "stage1_latest.pth"
STAGE1_BEST_CKPT = CHECKPOINT_DIR / "stage1_retrained_best.pth"
print(f"[CONFIG] Checkpoint directory: {CHECKPOINT_DIR.resolve()}")
print(f"[CONFIG] Latest checkpoint target: {STAGE1_LATEST_CKPT}")
print(f"[CONFIG] Best checkpoint target: {STAGE1_BEST_CKPT}")


---
## Step 2: Auto-Discover Codebase & Initialize Reproducible Environment
Searches for `src` directory across Google Drive, local paths, or falls back to GitHub clone.


In [ ]:
import os
import sys
from pathlib import Path
import random
import subprocess
import zipfile
import numpy as np
import torch

# Pure GitHub Codebase Discovery
CANDIDATES = [
    Path("/content/medical_wsss_repo"),
    Path("/content/medical-wsss"),
    Path.cwd(),
    Path.cwd().parent,
]

CODE_ROOT = next((str(p.resolve()) for p in CANDIDATES if (p / "src" / "models").is_dir()), None)
if CODE_ROOT is None:
    print("[GIT] Cloning official repository https://github.com/khoinguyen59/medical-wsss.git ...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/khoinguyen59/medical-wsss.git", "/content/medical_wsss_repo"], check=False)
    if Path("/content/medical_wsss_repo/src/models").is_dir():
        CODE_ROOT = str(Path("/content/medical_wsss_repo").resolve())
    else:
        CODE_ROOT = str(Path.cwd().resolve())

STAGE1_EXP_PRESET = os.environ.get("STAGE1_EXP_PRESET", "default").strip() or "default"
STAGE1_GIT_REF = os.environ.get("STAGE1_GIT_REF", "").strip()
if STAGE1_GIT_REF:
    if not (Path(CODE_ROOT) / ".git").exists():
        raise RuntimeError(f"STAGE1_GIT_REF={STAGE1_GIT_REF!r} requested, but {CODE_ROOT} is not a git repository")
    print(f"[GIT] Fetching & checking out STAGE1_GIT_REF={STAGE1_GIT_REF} in {CODE_ROOT} ...")
    _r_reset = subprocess.run(["git", "-C", CODE_ROOT, "reset", "--hard", "HEAD"], capture_output=True, text=True, check=False)
    if _r_reset.returncode != 0:
        raise RuntimeError(f"git reset --hard failed in {CODE_ROOT}: {_r_reset.stderr.strip()}")
    subprocess.run(["git", "-C", CODE_ROOT, "fetch", "origin", "+refs/heads/*:refs/remotes/origin/*", "--tags"], capture_output=True, text=True, check=False)
    subprocess.run(["git", "-C", CODE_ROOT, "fetch", "--depth", "1", "origin", STAGE1_GIT_REF], capture_output=True, text=True, check=False)
    _is_branch = subprocess.run(["git", "-C", CODE_ROOT, "rev-parse", "--verify", "--quiet", f"refs/remotes/origin/{STAGE1_GIT_REF}^{{commit}}"], capture_output=True, text=True, check=False)
    if _is_branch.returncode == 0:
        _rc_git = subprocess.run(["git", "-C", CODE_ROOT, "checkout", "-B", STAGE1_GIT_REF, f"refs/remotes/origin/{STAGE1_GIT_REF}"], capture_output=True, text=True, check=False)
        if _rc_git.returncode != 0:
            raise RuntimeError(f"Failed to checkout branch STAGE1_GIT_REF={STAGE1_GIT_REF!r}: {_rc_git.stderr.strip()}")
        subprocess.run(["git", "-C", CODE_ROOT, "pull", "--ff-only", "origin", STAGE1_GIT_REF], capture_output=True, text=True, check=False)
    else:
        _has_ref = subprocess.run(["git", "-C", CODE_ROOT, "rev-parse", "--verify", "--quiet", f"{STAGE1_GIT_REF}^{{commit}}"], capture_output=True, text=True, check=False)
        if _has_ref.returncode != 0:
            if (Path(CODE_ROOT) / ".git" / "shallow").exists():
                subprocess.run(["git", "-C", CODE_ROOT, "fetch", "--unshallow", "origin"], capture_output=True, text=True, check=False)
            else:
                subprocess.run(["git", "-C", CODE_ROOT, "fetch", "origin"], capture_output=True, text=True, check=False)
            subprocess.run(["git", "-C", CODE_ROOT, "fetch", "origin", STAGE1_GIT_REF], capture_output=True, text=True, check=False)
        _rc_git = subprocess.run(["git", "-C", CODE_ROOT, "checkout", "--force", STAGE1_GIT_REF], capture_output=True, text=True, check=False)
        if _rc_git.returncode != 0:
            raise RuntimeError(f"Failed to checkout STAGE1_GIT_REF={STAGE1_GIT_REF!r} in {CODE_ROOT}: {_rc_git.stderr.strip()}")
elif (Path(CODE_ROOT) / ".git").exists() and ("google.colab" in sys.modules):
    _head_br = subprocess.run(["git", "-C", CODE_ROOT, "symbolic-ref", "--short", "-q", "HEAD"], capture_output=True, text=True, check=False)
    if _head_br.returncode == 0 and _head_br.stdout.strip():
        subprocess.run(["git", "-C", CODE_ROOT, "fetch", "origin", _head_br.stdout.strip()], capture_output=True, text=True, check=False)
        subprocess.run(["git", "-C", CODE_ROOT, "pull", "--ff-only", "origin", _head_br.stdout.strip()], capture_output=True, text=True, check=False)

for _mod_k in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
    sys.modules.pop(_mod_k, None)
if CODE_ROOT in sys.path:
    sys.path.remove(CODE_ROOT)
sys.path.insert(0, CODE_ROOT)

_raw_seed = os.environ.get("STAGE1_SEED", "").strip()
SEED = int(_raw_seed) if _raw_seed else 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = torch.float16  # Native Turing Tensor Core FP16 acceleration for Tesla T4
print(f"[READY] Code root: {CODE_ROOT} | Preset={STAGE1_EXP_PRESET} | Seed={SEED} | GitRef={STAGE1_GIT_REF or 'HEAD'}")
print(f"[READY] Compute Device: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"[READY] Mixed Precision AMP Dtype: {AMP_DTYPE} (Tesla T4 Optimized)")


---
## Step 3: Resilient Dataset Download & Ingestion (23,422 Patches)
Downloads and extracts `BCSS-WSSS.zip` via HuggingFace mirror with high-speed mirror.


In [ ]:
import os
import sys
import zipfile
import urllib.request
from pathlib import Path

# Dataset locations
IN_COLAB = 'google.colab' in sys.modules if 'IN_COLAB' not in globals() else IN_COLAB
DATASET_DIR = Path('/content/BCSS-WSSS') if IN_COLAB else Path('./data/BCSS-WSSS')
DATASET_ZIP = Path('/content/BCSS-WSSS.zip') if IN_COLAB else Path('./data/BCSS-WSSS.zip')
DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)

HF_URL = 'https://huggingface.co/datasets/nttduc/wsss-pathology/resolve/main/BCSS-WSSS.zip'
GDRIVE_ID = '1sCK8Cqsf4yCHzzIx98vitocES2L_tkjn'

# Robust image presence check (searches recursively for any PNG patches)
def count_patches(folder):
    p = Path(folder)
    return len(list(p.rglob('*.png'))) if p.is_dir() else 0

# Check both author 'training/' and 'train/' folders for existing patches
total_train_patches = count_patches(DATASET_DIR / 'training') + count_patches(DATASET_DIR / 'train')
if total_train_patches < 100:
    if not DATASET_ZIP.is_file() or DATASET_ZIP.stat().st_size < 1000000:
        print(f'[DOWNLOAD] Downloading BCSS-WSSS dataset from HuggingFace mirror: {HF_URL}')
        try:
            urllib.request.urlretrieve(HF_URL, str(DATASET_ZIP))
            print(f'[DOWNLOAD] Download complete ({DATASET_ZIP.stat().st_size / (1024**3):.2f} GB)')
        except Exception as e:
            print(f'[WARN] HuggingFace mirror failed: {e}. Attempting gdown from Drive...')
            import subprocess
            subprocess.run(['pip', 'install', '-q', 'gdown'], check=False)
            subprocess.run(['gdown', '--id', GDRIVE_ID, '-O', str(DATASET_ZIP)], check=False)

    if DATASET_ZIP.is_file() and DATASET_ZIP.stat().st_size > 1000000:
        print(f'[EXTRACT] Extracting {DATASET_ZIP} to {DATASET_DIR} ...')
        with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
            zf.extractall(str(DATASET_DIR))
        print('[EXTRACT] Extraction finished successfully.')

# Auto-resolve TRAIN_DIR and VAL dirs supporting both author 'training/' and 'train/' conventions
def resolve_split_path(base_dir, split='train', sub=None):
    if split in ('train', 'training'):
        split_aliases = ['training', 'train']
    elif split in ('val', 'valid', 'validation'):
        split_aliases = ['val', 'valid', 'validation']
    else:
        split_aliases = [split]
    candidates = []
    for alias in split_aliases:
        if sub:
            candidates.extend([
                base_dir / alias / sub,
                base_dir / 'BCSS-WSSS' / alias / sub,
                Path('/content/BCSS-WSSS') / alias / sub,
                Path('/content') / alias / sub,
                Path('./data/BCSS-WSSS') / alias / sub,
                Path('./data') / alias / sub,
            ])
        else:
            candidates.extend([
                base_dir / alias,
                base_dir / alias / 'img',
                base_dir / 'BCSS-WSSS' / alias,
                base_dir / 'BCSS-WSSS' / alias / 'img',
                Path('/content/BCSS-WSSS') / alias,
                Path('/content/BCSS-WSSS') / alias / 'img',
                Path('/content') / alias,
                Path('/content') / alias / 'img',
                Path('./data/BCSS-WSSS') / alias,
                Path('./data') / alias,
            ])
    for c in candidates:
        if c.is_dir() and len(list(c.glob('*.png'))) > 0:
            return c
    for c in candidates:
        if c.is_dir() and len(list(c.rglob('*.png'))) > 0:
            return c
    default_alias = split_aliases[0]
    return (base_dir / default_alias / sub) if sub else (base_dir / default_alias)

TRAIN_DIR = resolve_split_path(DATASET_DIR, 'train')
VAL_IMG_DIR = resolve_split_path(DATASET_DIR, 'val', 'img')
VAL_MASK_DIR = resolve_split_path(DATASET_DIR, 'val', 'mask')
train_patches = list(TRAIN_DIR.glob('*.png')) if TRAIN_DIR.is_dir() else []
if len(train_patches) == 0 and TRAIN_DIR.is_dir():
    train_patches = list(TRAIN_DIR.rglob('*.png'))
val_patches = list(VAL_IMG_DIR.glob('*.png')) if VAL_IMG_DIR.is_dir() else []
if len(val_patches) == 0 and VAL_IMG_DIR.is_dir():
    val_patches = list(VAL_IMG_DIR.rglob('*.png'))
print(f'[DATASET] Active training directory: {TRAIN_DIR} (Patches: {len(train_patches):,})')
print(f'[DATASET] Active validation images: {VAL_IMG_DIR} (Patches: {len(val_patches):,})')
print(f'[DATASET] Active validation masks: {VAL_MASK_DIR}')


---
## Step 4: High-Performance Multi-Label Dataset & DataLoader Pipeline
Extracts weak multi-label diagnostic targets `[TUM STR LYM NEC]` from filenames with patient-level anti-leakage verification.


In [ ]:
import os
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

from src.config.stage1_experiments import resolve_stage1_config_from_env
from src.training.dataset_loader import (
    BCSSStage1TrainDataset,
    BCSSStage1ValDataset,
    RandomD4Transform,
    extract_patient_id,
    load_bcss_patient_splits,
)

EXP_CFG = resolve_stage1_config_from_env()
STAGE1_EXP_PRESET = EXP_CFG.preset_name
# Baseline Parity input normalization: 'zero_one' matches repos/WSSS-Tissue/1_train_stage1.py
# and domain-pretrained ResNet-38d checkpoints; metadata 'input_norm' is persisted for Stage 2.
STAGE1_INPUT_NORM = EXP_CFG.input_norm  # Resolved from preset / STAGE1_INPUT_NORM

# Data augmentation pipeline aligned with STAGE1_INPUT_NORM (no ColorJitter per author 1_train_stage1.py)
_train_ops = [transforms.Resize((224, 224))]
if EXP_CFG.use_d4_aug:
    _train_ops.append(RandomD4Transform())
else:
    _train_ops.extend([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
    ])
_train_ops.append(transforms.ToTensor())
_val_ops = [
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
]
if STAGE1_INPUT_NORM == 'imagenet':
    _norm_op = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    _train_ops.append(_norm_op)
    _val_ops.append(_norm_op)
train_transform = transforms.Compose(_train_ops)
val_transform = transforms.Compose(_val_ops)

BATCH_SIZE = (
    EXP_CFG.batch_size
    if (os.environ.get('STAGE1_BATCH_SIZE', '').strip() or EXP_CFG.preset_name != 'default')
    else 16
)
NUM_WORKERS = (
    EXP_CFG.num_workers
    if os.environ.get('STAGE1_NUM_WORKERS', '').strip()
    else 2
)
DRY_RUN = False if 'DRY_RUN' not in globals() else DRY_RUN  # Set to True for fast smoke-test
patient_splits = load_bcss_patient_splits()
train_patient_ids = patient_splits.get('train', [])
val_patient_ids = patient_splits.get('val', [])
assert set(train_patient_ids).isdisjoint(set(val_patient_ids)), 'Invariant 08 violation: train/val patient overlap!'

train_dataset = BCSSStage1TrainDataset(
    TRAIN_DIR,
    transform=train_transform,
    limit=64 if DRY_RUN else None,
    patient_ids=train_patient_ids,
    input_norm=STAGE1_INPUT_NORM,
    strict_patient_filter=True,
    seed=EXP_CFG.seed,
    use_d4_aug=EXP_CFG.use_d4_aug,
)
print(f'[DATASET] Stage 1 Train ingested {len(train_dataset):,} patches from {TRAIN_DIR} (preset={STAGE1_EXP_PRESET}, d4_aug={EXP_CFG.use_d4_aug})')
_dl_mp_kwargs = (
    {'persistent_workers': True, 'prefetch_factor': 2, 'multiprocessing_context': 'spawn'}
    if NUM_WORKERS > 0
    else {}
)
train_loader = DataLoader(
    train_dataset,
    batch_size=min(BATCH_SIZE, max(1, len(train_dataset))),
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=(len(train_dataset) >= BATCH_SIZE),
    **_dl_mp_kwargs,
)
val_dataset = BCSSStage1ValDataset(
    VAL_IMG_DIR,
    VAL_MASK_DIR,
    transform=val_transform,
    limit=32 if DRY_RUN else None,
    patient_ids=val_patient_ids,
    input_norm=STAGE1_INPUT_NORM,
    strict_patient_filter=True,
    seed=EXP_CFG.seed,
)
_uniq_val_pids = len({extract_patient_id(f.name) for f in val_dataset.files})
print(f'[DATASET] Stage 1 Val ingested {len(val_dataset):,} patches across {_uniq_val_pids} unique patients (Anti-Leakage Verified).')
val_loader = DataLoader(
    val_dataset,
    batch_size=min(BATCH_SIZE, max(1, len(val_dataset))),
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
    **_dl_mp_kwargs,
) if len(val_dataset) > 0 else None
print(f'[DATALOADER] Train batches/epoch: {len(train_loader)} (BS={BATCH_SIZE}, Samples={len(train_dataset):,}, norm={STAGE1_INPUT_NORM})')
print(f'[DATALOADER] Val batches/epoch: {len(val_loader) if val_loader else 0} (Samples={len(val_dataset):,})')


---
## Step 5: Instantiate Stage 1 ResNet38d Net_CAM & Progressive Dropout Attention (PDA)
Loads the official 105M-parameter ResNet-38d architecture with dilated convolutions and Net_CAM projection.


In [ ]:
import os
import sys
from pathlib import Path
import torch
import numpy as np

# Self-healing fallback definitions for DEVICE and AMP_DTYPE
if 'DEVICE' not in globals():
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if 'AMP_DTYPE' not in globals():
    AMP_DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 and torch.cuda.is_bf16_supported(including_emulation=False)) else torch.float16
if 'DRY_RUN' not in globals():
    DRY_RUN = False
if 'CHECKPOINT_DIR' not in globals():
    CHECKPOINT_DIR = Path('/content/checkpoints') if ('google.colab' in sys.modules) else Path('./checkpoints')

from src.config.stage1_experiments import resolve_stage1_config_from_env
if 'EXP_CFG' not in globals():
    EXP_CFG = resolve_stage1_config_from_env()
STAGE1_EXP_PRESET = EXP_CFG.preset_name
if 'STAGE1_INPUT_NORM' not in globals():
    STAGE1_INPUT_NORM = EXP_CFG.input_norm

try:
    from src.models.wsss_inference_methods import WSSSTissueStage1NetCAM
    print('[IMPORT] Successfully imported WSSSTissueStage1NetCAM from src.models')
except ImportError:
    from src.colab_pkg.model_loader import WSSSTissueStage1NetCAM
    print('[IMPORT] Imported WSSSTissueStage1NetCAM from src.colab_pkg')

from src.losses_metrics.losses import MultiPrototypePPCLoss
from src.training.train_stage1 import (
    AsymmetricLoss,
    AttentionModule,
    ModelEMA,
    ProjectionHead,
    Stage1Trainer,
    build_optimizer_and_scheduler,
    compute_multilabel_validation_metrics,
    load_pretrained_resnet38d_weights,
)

# Instantiate Stage 1 ResNet-38d Net_CAM
model = WSSSTissueStage1NetCAM(num_classes=4)

# Rule 17 Pretrained Weight Initialization Gate (NEVER silently train 105M ResNet-38d from scratch)
# Supports: local ilsvrc-cls_rna-a1_cls1000_ep-0001.params/.pth OR cross-organ LUAD histopathology ResNet-38d backbone (fc8 reset)
PRETRAINED_WEIGHTS_PATH = None  # Optional explicit path to ImageNet .params or .pth file
AUTO_DOWNLOAD_DOMAIN_PRETRAINED = True
ALLOW_RANDOM_INIT = DRY_RUN  # Strictly False for real training unless explicitly overridden

PRETRAINED_INFO = load_pretrained_resnet38d_weights(
    model,
    pretrained_path=PRETRAINED_WEIGHTS_PATH,
    allow_random_init=ALLOW_RANDOM_INIT,
    reset_classifier_head=True,
    auto_download_domain_pretrained=AUTO_DOWNLOAD_DOMAIN_PRETRAINED,
    cache_dir=CHECKPOINT_DIR,
)

model = model.to(DEVICE)
# Freeze BatchNorm2d and early stages (conv1a, b2, b2_1, b2_2) before building optimizer
model.train()
attention_module = AttentionModule(pda_mode=EXP_CFG.pda_mode).to(DEVICE)

# Optional Arm C training-only ProjectionHead + MultiPrototypePPCLoss (never exported into stage1_retrained_best.pth)
proj_head = None
mpppc_loss = None
if EXP_CFG.use_mpppc:
    proj_head = ProjectionHead(in_dim=4096, hidden_dim=256, out_dim=128, num_groups=16).to(DEVICE)
    mpppc_loss = MultiPrototypePPCLoss(
        num_classes=4,
        num_prototypes=EXP_CFG.ppc_num_prototypes,
        feat_dim=128,
        conf_thresh=EXP_CFG.ppc_conf_thresh,
        warm_start=True,
    ).to(DEVICE)

model_ema = ModelEMA(model, decay=EXP_CFG.ema_decay) if EXP_CFG.use_ema else None
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'[MODEL] Stage 1 ResNet38d Net_CAM: {total_params:,} params ({trainable_params:,} trainable) | Pretrained: {PRETRAINED_INFO["source"]} ({PRETRAINED_INFO["loaded_layers"]}/{PRETRAINED_INFO["total_backbone_layers"]} layers)')

# Parameterized Experiment Hyperparameters (resolved from STAGE1_EXP_PRESET / env overrides)
_has_env_epochs = bool(os.environ.get('STAGE1_MAX_EPOCHS', '').strip())
_cfg_epochs = EXP_CFG.max_epochs if (_has_env_epochs or EXP_CFG.preset_name != 'default') else 20
MAX_EPOCHS = 3 if DRY_RUN else _cfg_epochs
STOP_AFTER_EPOCH = None if DRY_RUN else EXP_CFG.stop_after_epoch
LEARNING_RATE = EXP_CFG.lr_bb
LR_FC8 = EXP_CFG.lr_fc8
WEIGHT_DECAY = EXP_CFG.wd_bb
WD_FC8 = EXP_CFG.wd_fc8
LOSS_TYPE = EXP_CFG.loss_type

_steps_per_ep = max(len(train_loader), 1) if 'train_loader' in globals() else 1
step_scheduler_per_batch = (EXP_CFG.scheduler_type == 'poly') or (EXP_CFG.warmup_epochs > 0.0)
warmup_steps = int(round(float(EXP_CFG.warmup_epochs) * _steps_per_ep)) if step_scheduler_per_batch else 0
total_steps = (MAX_EPOCHS * _steps_per_ep) if step_scheduler_per_batch else None

optimizer, scheduler = build_optimizer_and_scheduler(
    model,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    epochs=MAX_EPOCHS,
    optimizer_type=EXP_CFG.optimizer_type,
    scheduler_type=EXP_CFG.scheduler_type,
    lr_fc8=LR_FC8,
    wd_fc8=WD_FC8,
    momentum=EXP_CFG.momentum,
    warmup_steps=warmup_steps,
    total_steps=total_steps,
    poly_power=EXP_CFG.poly_power,
    eta_min=EXP_CFG.eta_min,
    extra_params=list(proj_head.parameters()) if proj_head is not None else None,
)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda' and AMP_DTYPE == torch.float16))
if LOSS_TYPE == 'asl':
    # Calibrated gentle ASL (gamma_neg=2.0, clip=0.02) to prevent minority-class threshold collapse
    criterion = AsymmetricLoss(gamma_neg=2.0, gamma_pos=0.0, clip=0.02)
elif LOSS_TYPE == 'bce':
    criterion = torch.nn.BCEWithLogitsLoss()
else:
    criterion = torch.nn.MultiLabelSoftMarginLoss()

_disp_steps = len(train_loader) if 'train_loader' in globals() else 0
print(f'[PRESET] Arm={STAGE1_EXP_PRESET} | Opt={EXP_CFG.optimizer_type} | Sched={EXP_CFG.scheduler_type} | PDA={EXP_CFG.pda_mode} | D4={EXP_CFG.use_d4_aug} | EMA={EXP_CFG.use_ema} ({EXP_CFG.ema_decay}) | Equiv={EXP_CFG.use_equiv_consistency} | MPPPC={EXP_CFG.use_mpppc}')
print(f'[PARITY] Loss={type(criterion).__name__} (LOSS_TYPE={LOSS_TYPE}) | norm={STAGE1_INPUT_NORM}')
print(f'[PARITY] Epochs={MAX_EPOCHS} (StopAfter={STOP_AFTER_EPOCH}) | lr_bb={LEARNING_RATE} lr_fc8={LR_FC8} wd_bb={WEIGHT_DECAY} wd_fc8={WD_FC8} | Steps/Epoch={_disp_steps} | Total Steps={_disp_steps * MAX_EPOCHS:,}')

def forward_stage1_dual(model, images, enable_pda=False, gamma=1.0, attn_module=None, active_labels=None):
    # Shared backbone forward pass; keep PDA + 4096-ch head in FP32 with autocast disabled
    conv6 = model.forward_backbone(images)  # (B, 4096, 28, 28)
    with torch.amp.autocast(device_type=conv6.device.type, enabled=False):
        conv6_f = conv6.float()
        fc8_w = model.fc8.weight.float()
        x_clean = model.dropout7(conv6_f)
        x_clean = model.pool(x_clean)
        logits_clean = torch.nn.functional.conv2d(x_clean, fc8_w).view(x_clean.size(0), -1)
        probs_clean = torch.sigmoid(logits_clean)
        if not enable_pda:
            return logits_clean, probs_clean, None, None
        mod = attn_module if attn_module is not None else globals().get('attention_module')
        if mod is None:
            mod = AttentionModule().to(conv6_f.device)
        conv6_pda = mod(conv6_f, fc8_w, gamma=gamma, active_labels=active_labels)
        x_pda = model.dropout7(conv6_pda)
        x_pda = model.pool(x_pda)
        logits_pda = torch.nn.functional.conv2d(x_pda, fc8_w).view(x_pda.size(0), -1)
        probs_pda = torch.sigmoid(logits_pda)
        return logits_clean, probs_clean, logits_pda, probs_pda

def forward_stage1_with_pda(model, images, enable_pda=False, gamma=1.0, attn_module=None, active_labels=None):
    logits_clean, probs_clean, logits_pda, probs_pda = forward_stage1_dual(
        model, images, enable_pda=enable_pda, gamma=gamma, attn_module=attn_module, active_labels=active_labels
    )
    if enable_pda and logits_pda is not None and probs_pda is not None:
        return logits_pda, probs_pda
    return logits_clean, probs_clean


---
## Step 6: Automatic Resume from Previous Epoch Checkpoint
Checks if a previous training run was interrupted, restoring optimizer and scheduler state seamlessly.


In [ ]:
import sys
from pathlib import Path
import torch
if 'DEVICE' not in globals():
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if 'CHECKPOINT_DIR' not in globals():
    CHECKPOINT_DIR = Path('/content/checkpoints') if ('google.colab' in sys.modules) else Path('./checkpoints')
if 'STAGE1_LATEST_CKPT' not in globals():
    STAGE1_LATEST_CKPT = CHECKPOINT_DIR / 'stage1_latest.pth'
if 'STAGE1_BEST_CKPT' not in globals():
    STAGE1_BEST_CKPT = CHECKPOINT_DIR / 'stage1_retrained_best.pth'
STAGE1_BEST_CLS_CKPT = CHECKPOINT_DIR / 'stage1_best_cls.pth'

start_epoch = 0
best_val_score = -1.0
best_cls_score = -1.0
best_val_metric_name = 'val_gated_pm_miou'
best_acc = 0.0
pda_gamma = float(EXP_CFG.initial_gamma) if 'EXP_CFG' in globals() else 1.0
epochs_without_improve = 0
joint_no_improve = 0
best_val_loss_seen = float('inf')
best_val_map_seen = -1.0
best_val_miou_seen = -1.0
selected_weights = 'raw'
_pending_scaler_state = None
history = {
    'epoch': [], 'loss': [], 'em': [], 'acc': [], 'lr': [],
    'val_loss': [], 'val_mAP': [], 'val_auroc': [], 'val_best_f1': [],
    'val_macro_f1': [], 'val_logits_mean': [],
    'val_gated_pm_miou': [], 'val_gated_pm_mdice': [],
    'val_gated_pm_miou_global': [], 'fc8_std': [],
    'ema_val_loss': [], 'ema_val_mAP': [], 'ema_val_auroc': [], 'ema_val_best_f1': [],
    'ema_val_gated_pm_miou': [], 'ema_val_gated_pm_miou_global': [], 'ema_fc8_std': [],
}

if STAGE1_LATEST_CKPT.is_file():
    print(f'[RESUME] Found existing checkpoint: {STAGE1_LATEST_CKPT}')
    _model_backup = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    try:
        ckpt = torch.load(STAGE1_LATEST_CKPT, map_location='cpu', weights_only=False)
        _ckpt_preset = str(ckpt.get('exp_preset', 'default') or 'default')
        _cur_preset = str(globals().get('STAGE1_EXP_PRESET', 'default') or 'default')
        if _ckpt_preset != _cur_preset:
            raise RuntimeError(f'exp_preset mismatch in checkpoint: {_ckpt_preset!r} != {_cur_preset!r}')
        state_dict = ckpt.get('state_dict', ckpt.get('model', ckpt))
        model.load_state_dict(state_dict, strict=False)
        if 'optimizer' in ckpt:
            optimizer.load_state_dict(ckpt['optimizer'])
            for _svs in optimizer.state.values():
                for _sk, _sv in _svs.items():
                    if isinstance(_sv, torch.Tensor) and _sk != 'step':
                        _svs[_sk] = _sv.to(DEVICE)
        if 'scheduler' in ckpt:
            scheduler.load_state_dict(ckpt['scheduler'])
        if globals().get('model_ema') is not None:
            if 'ema_state_dict' in ckpt:
                model_ema.load_state_dict(ckpt['ema_state_dict'])
            else:
                model_ema.shadow = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if globals().get('proj_head') is not None and 'proj_head_state_dict' in ckpt:
            proj_head.load_state_dict(ckpt['proj_head_state_dict'])
        if globals().get('mpppc_loss') is not None and 'mpppc_state_dict' in ckpt:
            mpppc_loss.load_state_dict(ckpt['mpppc_state_dict'])
        if isinstance(ckpt.get('scaler'), dict) and ckpt['scaler']:
            _pending_scaler_state = ckpt['scaler']
            if globals().get('scaler') is not None and scaler.is_enabled():
                scaler.load_state_dict(_pending_scaler_state)
        start_epoch = int(ckpt.get('epoch', -1)) + 1
        best_val_score = float(ckpt.get('best_val_score', ckpt.get('val_gated_pm_miou', -1.0)))
        best_cls_score = float(ckpt.get('best_cls_score', ckpt.get('val_mAP', -1.0)))
        best_val_metric_name = ckpt.get('best_val_metric', 'val_gated_pm_miou')
        best_acc = float(ckpt.get('best_acc', ckpt.get('macro_f1', 0.0)))
        pda_gamma = float(ckpt.get('pda_gamma', pda_gamma))
        epochs_without_improve = int(ckpt.get('epochs_without_improve', 0))
        joint_no_improve = int(ckpt.get('joint_no_improve', epochs_without_improve))
        best_val_loss_seen = float(ckpt.get('best_val_loss_seen', float('inf')))
        best_val_map_seen = float(ckpt.get('best_val_map_seen', best_cls_score))
        best_val_miou_seen = float(ckpt.get('best_val_miou_seen', best_val_score))
        selected_weights = str(ckpt.get('selected_weights', 'raw'))
        loaded_hist = ckpt.get('history', {})
        if isinstance(loaded_hist, dict):
            for k in history:
                if k in loaded_hist and isinstance(loaded_hist[k], list):
                    history[k] = list(loaded_hist[k])
        print(f'[RESUME] Resuming training from epoch {start_epoch}/{MAX_EPOCHS} (Best {best_val_metric_name}: {best_val_score:.4f}, best_cls={best_cls_score:.4f}, joint_no_improve={joint_no_improve})')
    except Exception as e:
        model.load_state_dict(_model_backup, strict=False)
        start_epoch = 0
        _pending_scaler_state = None
        print(f'[WARN] Failed to load checkpoint: {e}. Restored clean backbone for fresh training.')
else:
    print('[INIT] No previous checkpoint found. Starting Stage 1 training from initialized backbone.')


---
## Step 7: Multi-Epoch Training Loop with Interactive tqdm Progress Tracking
Trains ResNet38d with Progressive Dropout Attention (PDA) enabled after epoch 2 and automatic epoch checkpointing.


In [ ]:
import gc
import math
import sys
from pathlib import Path
import torch
import torch.nn.functional as F
import numpy as np
from tqdm.auto import tqdm

from src.config.stage1_experiments import resolve_stage1_config_from_env
from src.training.train_stage1 import Stage1Trainer

# Self-healing fallback definitions for DEVICE, AMP_DTYPE, and EXP_CFG
if 'DEVICE' not in globals():
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if 'AMP_DTYPE' not in globals():
    AMP_DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 and torch.cuda.is_bf16_supported(including_emulation=False)) else torch.float16
if 'EXP_CFG' not in globals():
    EXP_CFG = resolve_stage1_config_from_env()
if 'STAGE1_EXP_PRESET' not in globals():
    STAGE1_EXP_PRESET = EXP_CFG.preset_name
if 'pda_gamma' not in globals():
    pda_gamma = float(EXP_CFG.initial_gamma)
if 'epochs_without_improve' not in globals():
    epochs_without_improve = 0
if 'joint_no_improve' not in globals():
    joint_no_improve = 0
if 'best_cls_score' not in globals():
    best_cls_score = -1.0
if 'best_val_loss_seen' not in globals():
    best_val_loss_seen = float('inf')
if 'best_val_map_seen' not in globals():
    best_val_map_seen = -1.0
if 'best_val_miou_seen' not in globals():
    best_val_miou_seen = -1.0
if 'selected_weights' not in globals():
    selected_weights = 'raw'
if 'STAGE1_INPUT_NORM' not in globals():
    STAGE1_INPUT_NORM = EXP_CFG.input_norm
if 'LOSS_TYPE' not in globals():
    LOSS_TYPE = EXP_CFG.loss_type
if 'PRETRAINED_INFO' not in globals():
    PRETRAINED_INFO = {'source': 'unknown', 'loaded_layers': 0}
if 'CHECKPOINT_DIR' not in globals():
    CHECKPOINT_DIR = Path('/content/checkpoints') if ('google.colab' in sys.modules) else Path('./checkpoints')
if 'STAGE1_BEST_CLS_CKPT' not in globals():
    STAGE1_BEST_CLS_CKPT = CHECKPOINT_DIR / 'stage1_best_cls.pth'
if 'STOP_AFTER_EPOCH' not in globals():
    STOP_AFTER_EPOCH = None if globals().get('DRY_RUN', False) else EXP_CFG.stop_after_epoch

PDA_START_EPOCH = EXP_CFG.pda_start_epoch  # Default 3 -> activates at Epoch 4 after 3 warmup epochs

# Rule 17 Anti-Blind-Training Gate: require independent validation loop on non-dry-run
if not DRY_RUN and ('val_loader' not in globals() or val_loader is None or len(val_loader) == 0):
    raise RuntimeError('[STAGE1 ERROR] Independent val_loader is missing! Refusing blind training without validation (Rule 17).')

_step_sched_per_batch = globals().get(
    'step_scheduler_per_batch',
    (EXP_CFG.scheduler_type == 'poly') or (EXP_CFG.warmup_epochs > 0.0),
)
trainer = Stage1Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    device=DEVICE,
    enable_amp=(not DRY_RUN),
    pda_start_epoch=PDA_START_EPOCH,
    initial_gamma=pda_gamma,
    gamma_decay=EXP_CFG.gamma_decay,
    min_gamma=EXP_CFG.min_gamma,
    pda_mode=EXP_CFG.pda_mode,
    step_scheduler_per_batch=_step_sched_per_batch,
    model_ema=globals().get('model_ema'),
    proj_head=globals().get('proj_head'),
    mpppc_loss=globals().get('mpppc_loss'),
    lambda_ppc=EXP_CFG.lambda_ppc,
    ppc_start_epoch=EXP_CFG.ppc_start_epoch,
    use_equiv_consistency=EXP_CFG.use_equiv_consistency,
    equiv_scales=EXP_CFG.equiv_scales,
    lambda_eq_max=EXP_CFG.lambda_eq_max,
    eq_ramp_start_epoch=EXP_CFG.eq_ramp_start_epoch,
    eq_ramp_end_epoch=EXP_CFG.eq_ramp_end_epoch,
    amp_dtype=AMP_DTYPE,
)
if globals().get('_pending_scaler_state') and trainer.scaler.is_enabled():
    trainer.scaler.load_state_dict(globals()['_pending_scaler_state'])
    globals()['_pending_scaler_state'] = None

@torch.no_grad()
def evaluate_stage1_validation(model_eval, loader_eval):
    trainer.model = model_eval
    adapted_batches = ((imgs_v, lbls_v, gts_v, fns_v) for fns_v, imgs_v, lbls_v, gts_v in loader_eval)
    return trainer.evaluate(adapted_batches)

_trainable_all = list(model.parameters())
if globals().get('proj_head') is not None:
    _trainable_all.extend(list(proj_head.parameters()))

_loop_should_run = True
if not DRY_RUN and STOP_AFTER_EPOCH is not None and start_epoch >= int(STOP_AFTER_EPOCH):
    print(
        f'[STOP AFTER EPOCH] Start epoch {start_epoch+1} already past '
        f'STAGE1_STOP_AFTER_EPOCH={int(STOP_AFTER_EPOCH)} (LR horizon={MAX_EPOCHS}). Halting cleanly.',
        flush=True,
    )
    _loop_should_run = False

for epoch in (range(start_epoch, MAX_EPOCHS) if _loop_should_run else ()):
    model.train()
    if globals().get('proj_head') is not None:
        proj_head.train()
    if globals().get('mpppc_loss') is not None:
        mpppc_loss.train()

    running_loss = 0.0
    running_em = 0
    running_acc = 0.0
    total_samples = 0
    skipped_batches = 0

    enable_pda = (epoch >= PDA_START_EPOCH)
    lambda_eq = trainer.get_lambda_eq(epoch)
    enable_ppc = (
        trainer.proj_head is not None
        and trainer.mpppc_loss is not None
        and trainer.lambda_ppc > 0.0
        and epoch >= trainer.ppc_start_epoch
    )

    pbar = tqdm(
        train_loader,
        desc=f'Epoch {epoch+1:02d}/{MAX_EPOCHS:02d} [{STAGE1_EXP_PRESET} PDA={enable_pda}, gamma={pda_gamma:.3f}]',
        unit='batch',
        mininterval=10.0,
    )
    for step, (filenames, images, labels) in enumerate(pbar):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        batch_n = labels.size(0)

        optimizer.zero_grad(set_to_none=True)
        logits_clean, probs_clean, logits_pda, _, conv6_f = trainer.forward_dual_branches(
            images, enable_pda=enable_pda, targets=labels, return_conv6=True
        )
        with torch.amp.autocast(device_type=DEVICE.type, enabled=False):
            if enable_pda and logits_pda is not None:
                loss = 0.5 * criterion(logits_clean, labels) + 0.5 * criterion(logits_pda, labels)
            else:
                loss = criterion(logits_clean, labels)
            if trainer.use_equiv_consistency:
                loss_cls_t, loss_eq = trainer.compute_equivariant_branch(images, labels, conv6_f)
                loss = loss + 0.25 * loss_cls_t + lambda_eq * loss_eq
            if enable_ppc:
                proj_feats = trainer.proj_head(conv6_f)
                with torch.no_grad():
                    fc8_w_det = model.fc8.weight.float().detach()
                    cam_logits = F.conv2d(conv6_f.detach(), fc8_w_det)
                    raw_cams = F.relu(cam_logits)
                    cam_probs = F.softmax(cam_logits, dim=1)
                loss_ppc = trainer.mpppc_loss(
                    proj_feats, cam_probs, labels, raw_cams=raw_cams, rgb_images=images
                )
                loss = loss + trainer.lambda_ppc * loss_ppc

        if not torch.isfinite(loss):
            optimizer.zero_grad(set_to_none=True)
            skipped_batches += 1
            continue

        scale_before = trainer.scaler.get_scale() if trainer.scaler.is_enabled() else None
        trainer.scaler.scale(loss).backward()
        if trainer.scaler.is_enabled():
            trainer.scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(_trainable_all, max_norm=5.0)
        trainer.scaler.step(optimizer)
        trainer.scaler.update()

        step_executed = (not trainer.scaler.is_enabled()) or (
            scale_before is not None and trainer.scaler.get_scale() >= scale_before
        )
        if step_executed and globals().get('model_ema') is not None:
            model_ema.update(model)
        if step_executed and scheduler is not None and _step_sched_per_batch:
            scheduler.step()

        total_samples += batch_n
        running_loss += float(loss.detach().item()) * batch_n
        with torch.no_grad():
            pred_bin = probs_clean.detach() > 0.5
            gt_bin = labels.detach() > 0.5
            inter = (pred_bin & gt_bin).sum(dim=1).float()
            union = (pred_bin | gt_bin).sum(dim=1).float()
            acc_vec = torch.where(union > 0, inter / union.clamp(min=1.0), torch.ones_like(inter))
            running_em += int((pred_bin == gt_bin).all(dim=1).sum().item())
            running_acc += float(acc_vec.sum().item())

        if (step + 1) % 100 == 0 or (step + 1) == len(train_loader):
            current_vram = f'{torch.cuda.memory_allocated() / (1024**3):.1f}GB' if torch.cuda.is_available() else 'N/A'
            pbar.set_postfix({
                'loss': f'{running_loss / max(1, total_samples):.4f}',
                'acc': f'{running_acc / max(1, total_samples):.4f}',
                'lr': f'{optimizer.param_groups[0]["lr"]:.2e}',
                'vram': current_vram,
            })

    pbar.close()
    del pbar
    gc.collect()
    if scheduler is not None and not _step_sched_per_batch:
        scheduler.step()
    if pda_gamma > EXP_CFG.min_gamma:
        pda_gamma = max(EXP_CFG.min_gamma, pda_gamma * EXP_CFG.gamma_decay)
    trainer.gamma = pda_gamma

    epoch_loss = running_loss / max(1, total_samples)
    epoch_em = running_em / max(1, total_samples)
    epoch_acc = running_acc / max(1, total_samples)

    # Independent Validation Loop (Patient-Separated, Rule 17): evaluate BOTH raw and EMA weights
    raw_val_metrics = evaluate_stage1_validation(model, val_loader) if ('val_loader' in globals() and val_loader is not None and len(val_loader) > 0) else {}
    ema_val_metrics = {}
    if globals().get('model_ema') is not None and ('val_loader' in globals() and val_loader is not None and len(val_loader) > 0):
        with model_ema.swap_to_ema(model):
            ema_val_metrics = evaluate_stage1_validation(model, val_loader)
        trainer.model = model

    def _pick_primary_score(vm):
        if 'val_gated_pm_miou' in vm:
            return 'val_gated_pm_miou', float(vm['val_gated_pm_miou'])
        if 'val_mAP' in vm:
            return 'val_mAP', float(vm['val_mAP'])
        if 'val_macro_f1' in vm:
            return 'val_macro_f1', float(vm['val_macro_f1'])
        return 'train_macro_f1_dry_run', float(epoch_acc)

    best_val_metric_name, raw_score = _pick_primary_score(raw_val_metrics)
    _, ema_score = _pick_primary_score(ema_val_metrics) if ema_val_metrics else (best_val_metric_name, -1.0)
    if globals().get('model_ema') is not None and ema_val_metrics and ema_score >= raw_score:
        selected_weights = 'ema'
        val_metrics = ema_val_metrics
        current_val_score = ema_score
        winning_state_dict = model_ema.clean_model_state_dict()
    else:
        selected_weights = 'raw'
        val_metrics = raw_val_metrics
        current_val_score = raw_score
        winning_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    raw_cls_score = float(raw_val_metrics.get('val_mAP', raw_val_metrics.get('val_macro_f1', epoch_acc)))
    ema_cls_score = float(ema_val_metrics.get('val_mAP', ema_val_metrics.get('val_macro_f1', -1.0))) if ema_val_metrics else -1.0
    if globals().get('model_ema') is not None and ema_val_metrics and ema_cls_score >= raw_cls_score:
        cls_selected_weights = 'ema'
        current_cls_score = ema_cls_score
        cls_state_dict = model_ema.clean_model_state_dict()
        cls_metrics_ref = ema_val_metrics
    else:
        cls_selected_weights = 'raw'
        current_cls_score = raw_cls_score
        cls_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        cls_metrics_ref = raw_val_metrics

    val_loss = float(raw_val_metrics.get('val_loss', 0.0))
    val_map = float(raw_val_metrics.get('val_mAP', 0.0))
    val_auroc = float(raw_val_metrics.get('val_auroc', 0.0))
    val_best_f1 = float(raw_val_metrics.get('val_best_f1', 0.0))
    val_f1_05 = float(raw_val_metrics.get('val_macro_f1', 0.0))
    val_logits_mean = float(raw_val_metrics.get('val_logits_mean', 0.0))
    val_pm_miou = float(raw_val_metrics.get('val_gated_pm_miou', 0.0))
    val_pm_mdice = float(raw_val_metrics.get('val_gated_pm_mdice', 0.0))
    val_pm_miou_g = float(raw_val_metrics.get('val_gated_pm_miou_global', 0.0))
    fc8_std = float(raw_val_metrics.get('fc8_std', 0.0))
    fc8_norms = [
        round(float(raw_val_metrics.get('fc8_norm_tum', 0.0)), 4),
        round(float(raw_val_metrics.get('fc8_norm_str', 0.0)), 4),
        round(float(raw_val_metrics.get('fc8_norm_lym', 0.0)), 4),
        round(float(raw_val_metrics.get('fc8_norm_nec', 0.0)), 4),
    ]
    pm_ious_ch = [
        round(float(raw_val_metrics.get('val_pm_iou_tum', 0.0)), 4),
        round(float(raw_val_metrics.get('val_pm_iou_str', 0.0)), 4),
        round(float(raw_val_metrics.get('val_pm_iou_lym', 0.0)), 4),
        round(float(raw_val_metrics.get('val_pm_iou_nec', 0.0)), 4),
    ]
    pm_ious_g = [
        round(float(raw_val_metrics.get('val_pm_iou_global_tum', 0.0)), 4),
        round(float(raw_val_metrics.get('val_pm_iou_global_str', 0.0)), 4),
        round(float(raw_val_metrics.get('val_pm_iou_global_lym', 0.0)), 4),
        round(float(raw_val_metrics.get('val_pm_iou_global_nec', 0.0)), 4),
    ]

    history['epoch'].append(epoch + 1)
    history['loss'].append(round(epoch_loss, 5))
    history['em'].append(round(epoch_em, 5))
    history['acc'].append(round(epoch_acc, 5))
    history['lr'].append(optimizer.param_groups[0]['lr'])
    history['val_loss'].append(round(val_loss, 5))
    history['val_mAP'].append(round(val_map, 4))
    history['val_auroc'].append(round(val_auroc, 4))
    history['val_best_f1'].append(round(val_best_f1, 4))
    history['val_macro_f1'].append(round(val_f1_05, 4))
    history['val_logits_mean'].append(round(val_logits_mean, 4))
    history['val_gated_pm_miou'].append(round(val_pm_miou, 4))
    history['val_gated_pm_mdice'].append(round(val_pm_mdice, 4))
    history['val_gated_pm_miou_global'].append(round(val_pm_miou_g, 4))
    history['fc8_std'].append(round(fc8_std, 6))
    if ema_val_metrics:
        history['ema_val_loss'].append(round(float(ema_val_metrics.get('val_loss', 0.0)), 5))
        history['ema_val_mAP'].append(round(float(ema_val_metrics.get('val_mAP', 0.0)), 4))
        history['ema_val_auroc'].append(round(float(ema_val_metrics.get('val_auroc', 0.0)), 4))
        history['ema_val_best_f1'].append(round(float(ema_val_metrics.get('val_best_f1', 0.0)), 4))
        history['ema_val_gated_pm_miou'].append(round(float(ema_val_metrics.get('val_gated_pm_miou', 0.0)), 4))
        history['ema_val_gated_pm_miou_global'].append(round(float(ema_val_metrics.get('val_gated_pm_miou_global', 0.0)), 4))
        history['ema_fc8_std'].append(round(float(ema_val_metrics.get('fc8_std', 0.0)), 6))

    print(
        f'[EPOCH {epoch+1:02d}] ({STAGE1_EXP_PRESET}) Train Loss={epoch_loss:.4f} EM={epoch_em:.4f} Acc={epoch_acc:.4f} | '
        f'ValLoss={val_loss:.4f} Val mAP={val_map:.4f} AUROC={val_auroc:.4f} BestF1={val_best_f1:.4f} '
        f'F1@0.5={val_f1_05:.4f} LogitMean={val_logits_mean:+.3f} '
        f'GatedPM_mIoU={val_pm_miou:.4f} (Global={val_pm_miou_g:.4f}, Dice={val_pm_mdice:.4f}, IoUs={pm_ious_ch}, GlobalIoUs={pm_ious_g}) | '
        f'fc8_std={fc8_std:.5f} fc8_norms={fc8_norms}',
        flush=True,
    )
    if ema_val_metrics:
        print(
            f'  [EMA] ValLoss={float(ema_val_metrics.get("val_loss", 0.0)):.4f} '
            f'Val mAP={float(ema_val_metrics.get("val_mAP", 0.0)):.4f} '
            f'AUROC={float(ema_val_metrics.get("val_auroc", 0.0)):.4f} '
            f'BestF1={float(ema_val_metrics.get("val_best_f1", 0.0)):.4f} '
            f'GatedPM_mIoU={float(ema_val_metrics.get("val_gated_pm_miou", 0.0)):.4f} '
            f'(Global={float(ema_val_metrics.get("val_gated_pm_miou_global", 0.0)):.4f}) | '
            f'fc8_std={float(ema_val_metrics.get("fc8_std", 0.0)):.5f} | selected={selected_weights}',
            flush=True,
        )

    if total_samples > 0 and np.isfinite(epoch_loss):
        # Select best checkpoint strictly by independent validation metric across raw & EMA
        improved = (current_val_score >= best_val_score)
        if improved:
            best_val_score = current_val_score
            best_acc = max(best_acc, float(val_metrics.get('val_best_f1', val_best_f1)))
            epochs_without_improve = 0
        else:
            epochs_without_improve += 1

        cls_improved = (current_cls_score >= best_cls_score)
        if cls_improved:
            best_cls_score = current_cls_score

        # Joint early-stop patience across {val_gated_pm_miou, val_mAP, val_loss}
        cur_miou_j = float(val_metrics.get('val_gated_pm_miou', -1.0))
        cur_map_j = float(val_metrics.get('val_mAP', -1.0))
        cur_vloss_j = float(val_metrics.get('val_loss', float('inf')))
        any_joint_improve = False
        if cur_miou_j > best_val_miou_seen + 1e-6:
            best_val_miou_seen = cur_miou_j
            any_joint_improve = True
        if cur_map_j > best_val_map_seen + 1e-6:
            best_val_map_seen = cur_map_j
            any_joint_improve = True
        if cur_vloss_j < best_val_loss_seen - 1e-6:
            best_val_loss_seen = cur_vloss_j
            any_joint_improve = True
        joint_no_improve = 0 if any_joint_improve else (joint_no_improve + 1)

        latest_state = {
            'epoch': epoch,
            'exp_preset': STAGE1_EXP_PRESET,
            'selected_weights': selected_weights,
            'state_dict': {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'scaler': trainer.scaler.state_dict(),
            'input_norm': STAGE1_INPUT_NORM,
            'loss_type': LOSS_TYPE,
            'pretrained_source': PRETRAINED_INFO.get('source', 'unknown'),
            'pretrained_loaded_layers': PRETRAINED_INFO.get('loaded_layers', 0),
            'best_val_score': best_val_score,
            'best_cls_score': best_cls_score,
            'best_val_metric': best_val_metric_name,
            'best_acc': best_acc,
            'best_val_loss_seen': best_val_loss_seen,
            'best_val_map_seen': best_val_map_seen,
            'best_val_miou_seen': best_val_miou_seen,
            'epochs_without_improve': epochs_without_improve,
            'joint_no_improve': joint_no_improve,
            'macro_f1': float(val_metrics.get('val_best_f1', val_best_f1)),
            'val_mAP': float(val_metrics.get('val_mAP', val_map)),
            'val_auroc': float(val_metrics.get('val_auroc', val_auroc)),
            'val_best_f1': float(val_metrics.get('val_best_f1', val_best_f1)),
            'val_macro_f1': float(val_metrics.get('val_macro_f1', val_f1_05)),
            'val_logits_mean': float(val_metrics.get('val_logits_mean', val_logits_mean)),
            'val_gated_pm_miou': float(val_metrics.get('val_gated_pm_miou', val_pm_miou)),
            'val_gated_pm_mdice': float(val_metrics.get('val_gated_pm_mdice', val_pm_mdice)),
            'val_gated_pm_miou_global': float(val_metrics.get('val_gated_pm_miou_global', val_pm_miou_g)),
            'fc8_std': float(val_metrics.get('fc8_std', fc8_std)),
            'val_metrics': val_metrics,
            'pda_gamma': pda_gamma,
            'history': history,
        }
        if globals().get('model_ema') is not None:
            latest_state['ema_state_dict'] = model_ema.state_dict()
        if globals().get('proj_head') is not None:
            latest_state['proj_head_state_dict'] = {k: v.detach().cpu().clone() for k, v in proj_head.state_dict().items()}
        if globals().get('mpppc_loss') is not None:
            latest_state['mpppc_state_dict'] = {k: v.detach().cpu().clone() for k, v in mpppc_loss.state_dict().items()}
        torch.save(latest_state, STAGE1_LATEST_CKPT)
        print(f'[CHECKPOINT] Saved epoch {epoch+1} to {STAGE1_LATEST_CKPT} (input_norm={STAGE1_INPUT_NORM})', flush=True)

        if improved or not STAGE1_BEST_CKPT.is_file():
            best_state = {
                k: v for k, v in latest_state.items()
                if k not in ('optimizer', 'scheduler', 'ema_state_dict', 'proj_head_state_dict', 'mpppc_state_dict')
            }
            best_state['state_dict'] = winning_state_dict
            torch.save(best_state, STAGE1_BEST_CKPT)
            print(f'[CHECKPOINT] >>> New best {best_val_metric_name}={best_val_score:.4f} (selected={selected_weights}, val_mAP={float(val_metrics.get("val_mAP", val_map)):.4f})! Saved compact best ckpt to {STAGE1_BEST_CKPT}', flush=True)

        if cls_improved or not STAGE1_BEST_CLS_CKPT.is_file():
            cls_state = {
                'epoch': epoch,
                'exp_preset': STAGE1_EXP_PRESET,
                'selected_weights': cls_selected_weights,
                'best_val_metric': 'val_mAP',
                'best_val_score': best_cls_score,
                'val_mAP': float(cls_metrics_ref.get('val_mAP', 0.0)),
                'val_auroc': float(cls_metrics_ref.get('val_auroc', 0.0)),
                'val_best_f1': float(cls_metrics_ref.get('val_best_f1', 0.0)),
                'val_gated_pm_miou': float(cls_metrics_ref.get('val_gated_pm_miou', 0.0)),
                'val_gated_pm_miou_global': float(cls_metrics_ref.get('val_gated_pm_miou_global', 0.0)),
                'fc8_std': float(cls_metrics_ref.get('fc8_std', 0.0)),
                'input_norm': STAGE1_INPUT_NORM,
                'loss_type': LOSS_TYPE,
                'state_dict': cls_state_dict,
            }
            torch.save(cls_state, STAGE1_BEST_CLS_CKPT)
            print(f'[CHECKPOINT] >>> New best cls val_mAP={best_cls_score:.4f} (selected={cls_selected_weights})! Saved compact cls ckpt to {STAGE1_BEST_CLS_CKPT}', flush=True)

        # Fatal Calibration Collapse Guard vs Clean Early Stop Patience
        if not DRY_RUN:
            sel_best_f1 = float(val_metrics.get('val_best_f1', 1.0))
            sel_val_loss = float(val_metrics.get('val_loss', 0.0))
            sel_val_map = float(val_metrics.get('val_mAP', 1.0))
            is_nan = (not math.isfinite(sel_val_loss)) or (not math.isfinite(epoch_loss))
            is_collapse = (epoch >= 4) and (sel_best_f1 < 0.55 or sel_val_loss > 1.20 or sel_val_map < 0.65)
            if is_nan or is_collapse:
                print(
                    f'[CALIBRATION COLLAPSE ALARM] Epoch {epoch+1}: Calibration/collapse detected '
                    f'(val_best_f1={sel_best_f1:.4f}, val_loss={sel_val_loss:.4f}, val_mAP={sel_val_map:.4f}, logits_mean={val_logits_mean:+.3f}). Halting.',
                    flush=True,
                )
                break
            if STOP_AFTER_EPOCH is not None and (epoch + 1) >= int(STOP_AFTER_EPOCH):
                print(
                    f'[STOP AFTER EPOCH] Epoch {epoch+1}: Completed target STAGE1_STOP_AFTER_EPOCH={int(STOP_AFTER_EPOCH)} '
                    f'(LR horizon={MAX_EPOCHS}). Halting cleanly.',
                    flush=True,
                )
                break
            if (epoch + 1) >= EXP_CFG.min_epochs_early_stop and joint_no_improve >= EXP_CFG.early_stop_patience:
                print(
                    f'[EARLY STOP PATIENCE] Epoch {epoch+1}: No joint improvement across '
                    f'{{val_gated_pm_miou, val_mAP, val_loss}} for {joint_no_improve} epochs '
                    f'(min_epochs={EXP_CFG.min_epochs_early_stop}, best {best_val_metric_name}={best_val_score:.4f}). Halting.',
                    flush=True,
                )
                break
    else:
        raise RuntimeError(f'Non-finite epoch loss at epoch {epoch+1}: {epoch_loss}')


---
## Step 8: Plot Training Curves & Export Deliverables ZIP
Generates 300 DPI loss/accuracy curves, exports JSON metrics, and triggers automatic browser download.


In [ ]:
import os
import sys
import json
import shutil
import zipfile
from pathlib import Path
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules if 'IN_COLAB' not in globals() else IN_COLAB
CHECKPOINT_DIR = Path('/content/checkpoints') if IN_COLAB else Path('./checkpoints')
STAGE1_BEST_CKPT = CHECKPOINT_DIR / 'stage1_retrained_best.pth' if 'STAGE1_BEST_CKPT' not in globals() else STAGE1_BEST_CKPT
STAGE1_BEST_CLS_CKPT = CHECKPOINT_DIR / 'stage1_best_cls.pth' if 'STAGE1_BEST_CLS_CKPT' not in globals() else STAGE1_BEST_CLS_CKPT
STAGE1_LATEST_CKPT = CHECKPOINT_DIR / 'stage1_latest.pth' if 'STAGE1_LATEST_CKPT' not in globals() else STAGE1_LATEST_CKPT
STAGE1_INPUT_NORM = globals().get('STAGE1_INPUT_NORM', 'zero_one')
STAGE1_EXP_PRESET = globals().get('STAGE1_EXP_PRESET', 'default')

OUTPUT_DIR = Path('/content/stage1_results') if IN_COLAB else Path('./stage1_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Bundle compact best checkpoints into output package (keep resume-only latest ckpt on disk to avoid 3x ZIP bloat)
for _ckpt_p in (STAGE1_BEST_CKPT, STAGE1_BEST_CLS_CKPT):
    if _ckpt_p.is_file():
        shutil.copy(_ckpt_p, OUTPUT_DIR / _ckpt_p.name)
        print(f'[PACKAGE] Bundled checkpoint {_ckpt_p.name} ({_ckpt_p.stat().st_size / (1024**2):.1f} MB)')
if STAGE1_LATEST_CKPT.is_file():
    print(f'[RESUME] Latest optimizer checkpoint retained on disk at {STAGE1_LATEST_CKPT} ({STAGE1_LATEST_CKPT.stat().st_size / (1024**2):.1f} MB)')

# Plot training loss and independent validation curves
if history.get('epoch'):
    fig, ax1 = plt.subplots(figsize=(9, 5), dpi=300)
    ax1.set_xlabel('Epoch', fontweight='bold')
    ax1.set_ylabel('Loss', color='tab:red', fontweight='bold')
    ax1.plot(history['epoch'], history['loss'], color='tab:red', marker='o', label='Train Loss')
    if history.get('val_loss'):
        ax1.plot(history['epoch'], history['val_loss'], color='tab:orange', linestyle='--', marker='.', label='Val Loss')
    ax1.tick_params(axis='y', labelcolor='tab:red')
    ax1.grid(True, linestyle='--', alpha=0.5)

    ax2 = ax1.twinx()
    ax2.set_ylabel('Validation Metrics (mAP / AUROC / PM mIoU)', color='tab:blue', fontweight='bold')
    if history.get('val_mAP'):
        ax2.plot(history['epoch'], history['val_mAP'], color='tab:blue', marker='s', label='Val mAP')
    if history.get('val_auroc'):
        ax2.plot(history['epoch'], history['val_auroc'], color='tab:purple', marker='d', label='Val AUROC')
    if history.get('val_gated_pm_miou'):
        ax2.plot(history['epoch'], history['val_gated_pm_miou'], color='tab:green', marker='^', label='Val Gated PM mIoU')
    if history.get('ema_val_gated_pm_miou') and len(history['ema_val_gated_pm_miou']) == len(history['epoch']):
        ax2.plot(history['epoch'], history['ema_val_gated_pm_miou'], color='tab:cyan', linestyle=':', marker='v', label='EMA Gated PM mIoU')
    ax2.tick_params(axis='y', labelcolor='tab:blue')

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right', fontsize=8)
    plt.title(f'BCSS-WSSS Stage 1 ResNet38d ({STAGE1_EXP_PRESET}) Training & Validation Curves', fontweight='bold')
    fig.tight_layout()
    plot_path = OUTPUT_DIR / 'stage1_training_curves.png'
    plt.savefig(plot_path, dpi=300)
    plt.close()
    print(f'[VISUALIZATION] Saved training plot to {plot_path}')

# Save metrics JSON
_ema_latest_metrics = {
    k: history[k][-1]
    for k in (
        'ema_val_loss', 'ema_val_mAP', 'ema_val_auroc', 'ema_val_best_f1',
        'ema_val_gated_pm_miou', 'ema_val_gated_pm_miou_global', 'ema_fc8_std',
    )
    if history.get(k)
}
_final_fc8_std = float(history['fc8_std'][-1]) if history.get('fc8_std') else 0.0
metrics_path = OUTPUT_DIR / 'STAGE1_TRAINING_METRICS.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump({
        'exp_preset': STAGE1_EXP_PRESET,
        'selected_weights': globals().get('selected_weights', 'raw'),
        'best_val_metric': globals().get('best_val_metric_name', 'val_gated_pm_miou'),
        'best_val_score': globals().get('best_val_score', 0.0),
        'best_cls_score': globals().get('best_cls_score', 0.0),
        'best_accuracy': globals().get('best_acc', 0.0),
        'fc8_std': _final_fc8_std,
        'ema_metrics': _ema_latest_metrics,
        'input_norm': STAGE1_INPUT_NORM,
        'loss_type': globals().get('LOSS_TYPE', 'soft_margin'),
        'max_epochs': globals().get('MAX_EPOCHS', EXP_CFG.max_epochs if 'EXP_CFG' in globals() else 25),
        'stop_after_epoch': globals().get('STOP_AFTER_EPOCH', None),
        'exp_config': globals()['EXP_CFG'].to_dict() if 'EXP_CFG' in globals() else {},
        'pretrained_info': globals().get('PRETRAINED_INFO', {}),
        'history': history,
    }, f, indent=2)
print(f'[METRICS] Saved metrics JSON to {metrics_path}')

# Compress results and trigger download in Colab
ZIP_EXPORT = Path('/content/SE374_Stage1_Training_Results.zip') if IN_COLAB else Path('./SE374_Stage1_Training_Results.zip')
with zipfile.ZipFile(ZIP_EXPORT, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in OUTPUT_DIR.glob('*'):
        if f.is_file():
            zf.write(f, arcname=f.name)
print(f'[PACKAGE] Compressed results to {ZIP_EXPORT} ({ZIP_EXPORT.stat().st_size / (1024**2):.1f} MB)')

if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(ZIP_EXPORT))
        print('[DOWNLOAD] Triggered browser download of results ZIP.')
    except Exception as e:
        print(f'[NOTICE] Browser download skipped: {e}')
